In [6]:
import numpy as np
import pandas as pd


n = 1000
np.random.seed(0)

# Grundlegende Einkommens- und Altersdaten erzeugen, Einkommen korreliert mit Alter
alter = np.random.normal(40, 10, n)
einkommen = np.random.normal(50000 + 200 * (alter - 40), 15000, n)  

# Kreditscore, teilweise abhängig von Einkommen
basis_kreditscore = np.random.normal(0, 1, n)
kreditscore = 850 - (500 - 2 * (einkommen / 1000)) + 50 * basis_kreditscore

# Weitere Merkmale
berufsstatus = np.random.choice(['angestellt', 'selbstständig', 'arbeitslos'], n, p=[0.7, 0.2, 0.1])
wohnsituation = np.random.choice(['Eigentum', 'Miete'], n, p=[0.6, 0.4])
anzahl_kredite = np.random.poisson(2, n)
bildungsniveau = np.random.choice(['Hochschulabschluss', 'Abitur', 'Hauptschulabschluss'], n, p=[0.4, 0.4, 0.2])

# Zufälliger Risikofaktor
zufaelliges_risiko = np.random.binomial(1, 0.1, n)  # 10% Chance, unerwartet als risikoreich eingestuft zu werden

# Risikobewertungskriterien
risiko = zufaelliges_risiko  # 10% Baseline-Risiko aufgrund zufälliger Faktoren
risiko |= (kreditscore < 550)  # Anheben des Schwellenwerts für Kreditscore
risiko |= ((einkommen < 25000) & (alter > 50))  # Striktere Bedingung für Einkommen und Alter
risiko |= (anzahl_kredite > 5)  # Anheben des Schwellenwerts für die Anzahl der Kredite
risiko |= ((alter < 25) & (berufsstatus == 'selbstständig') & (einkommen < 30000))  # Jüngere Selbstständige mit sehr niedrigem Einkommen
risiko |= ((wohnsituation == 'Miete') & (anzahl_kredite >= 4) & (bildungsniveau == 'Hauptschulabschluss'))  # Erhöhung der Kreditanzahl für Mieter mit niedriger Bildung



# DataFrame erstellen
daten = pd.DataFrame({
    'Alter': alter,
    'Einkommen': einkommen,
    'Kreditscore': kreditscore,
    'Berufsstatus': berufsstatus,
    'Wohnsituation': wohnsituation,
    'Anzahl Kredite': anzahl_kredite,
    'Bildungsniveau': bildungsniveau,
    'Zufälliges Risiko': zufaelliges_risiko,
    'Risiko': risiko
})

# Daten anzeigen
print(daten.head())


       Alter     Einkommen  Kreditscore   Berufsstatus Wohnsituation  \
0  57.640523  61867.544888   397.089037     angestellt      Eigentum   
1  44.001572  64187.422727   392.776337  selbstständig      Eigentum   
2  49.787380  45622.753606   443.552260     arbeitslos      Eigentum   
3  62.408932  56052.496840   414.186270     angestellt      Eigentum   
4  58.675580  57155.915857   460.271251     angestellt      Eigentum   

   Anzahl Kredite       Bildungsniveau  Zufälliges Risiko  Risiko  
0               1               Abitur                  1       1  
1               0   Hochschulabschluss                  1       1  
2               0   Hochschulabschluss                  1       1  
3               1  Hauptschulabschluss                  1       1  
4               3   Hochschulabschluss                  1       1  


In [44]:
def inf_JK_bagged_variance(N_bi, tree_predictions_b, chunk_size=1):
    """Calculates the infinitesimal jackknife variance estimate."""
    B, n_data_points = N_bi.shape
    n_preds = tree_predictions_b.shape[1]

    N_star_mean = 7*np.ones((n_data_points,n_preds)).astype(np.float32)
    print(N_star_mean)
    T_N_star_mean = np.mean(tree_predictions_b, axis=0).astype(np.float32)

    # Initialize the covariance matrix
    cov_matrix = np.zeros((n_data_points, n_preds), dtype=np.float32)

    # Calculate in chunks to avoid memory issues
    for start in range(0, B, chunk_size):
        end = min(start + chunk_size, B)
        chunk_N_bi = N_bi[start:end]
        print(chunk_N_bi)
        chunk_tree_predictions_b = tree_predictions_b[start:end]
        print(f'\ndiff_start{start}: ')
        print('chunks:')
        print(chunk_N_bi[:, :, None] )
        print(chunk_N_bi[:, :, None].shape)
        #print('\n')
        print('differenzen: ')
        print(chunk_N_bi[:, :, None] - N_star_mean)
        
        cov_matrix += np.sum(
            (chunk_N_bi[:, :, None] - 2154)
            * (chunk_tree_predictions_b[:, None, :] - T_N_star_mean),
            axis=0,
        ).astype(np.float32)
        print('cov_matrix:')
        print(cov_matrix)

    cov_matrix /= (B-1)

    bias_correction = (((n_data_points-1)*B) / (B-1)**3) * np.sum(
        (tree_predictions_b - T_N_star_mean) ** 2, axis=0
    ).astype(np.float32)

    # Calculate infinitesimal jackknife estimate
    bagged_inf_jackknife_est = np.sum(cov_matrix**2, axis=0) 

    return bagged_inf_jackknife_est

from utils import create_bootstrap_indices_and_Nbi
#from figure2_wager import 
_, nbi = create_bootstrap_indices_and_Nbi(2,3,4)

tree_preds = np.array([[6,8],[10,15],[5,7]])

inf_JK_bagged_variance(nbi, tree_preds)

[[7. 7.]
 [7. 7.]]
[[2 0]]

diff_start0: 
chunks:
[[[2]
  [0]]]
(1, 2, 1)
differenzen: 
[[[-5. -5.]
  [-7. -7.]]]
cov_matrix:
[[2152. 4304.]
 [2154. 4308.]]
[[0 2]]

diff_start1: 
chunks:
[[[0]
  [2]]]
(1, 2, 1)
differenzen: 
[[[-7. -7.]
  [-5. -5.]]]
cov_matrix:
[[-4310. -6466.]
 [-4302. -6452.]]
[[1 1]]

diff_start2: 
chunks:
[[[1]
  [1]]]
(1, 2, 1)
differenzen: 
[[[-6. -6.]
  [-6. -6.]]]
cov_matrix:
[[-4. -7.]
 [ 4.  7.]]


array([ 8. , 24.5], dtype=float32)

In [110]:
import numpy as np

def inf_JK_bagged_variance(N_bi, T_N_b):

    B, n_data_points = N_bi.shape
    n_preds = T_N_b.shape[1]

    T_N_star_mean = np.mean(T_N_b, axis=0).astype(np.float32)

    # Initialize the covariance vector
    cov_i = np.zeros((n_data_points, n_preds), dtype=np.float32)

    for i in range(n_data_points):
        for pred in range(n_preds):
            
            cov_i[i, pred] = np.dot(
                (N_bi[:, i] - 165) , (T_N_b[:, pred] - T_N_star_mean[pred])
            ).astype(np.float32) / (B - 1)
    
    print(cov_i)

    # Calculate the infinitesimal jackknife variance estimate
    var_IJK_U = np.sum(cov_i**2, axis=0) 

    bias_correction = (((n_data_points-1)*B) / (B-1)**3) * np.var(
        tree_predictions_b, axis=0
    ).astype(np.float32)
    return var_IJK_U

# Test the function with example data
def test_inf_JK_bagged_variance():
    N_bi = np.array([[2, 0],
                     [0, 2],
                     [1, 1]])
    T_N_b = np.array([[6, 8],
                      [10, 15],
                      [5, 7]])

    var_IJK_U = inf_JK_bagged_variance(N_bi, T_N_b)
    print("Infinitesimale Jackknife-Varianzschätzung:", var_IJK_U)

# Run the test function
test_inf_JK_bagged_variance()



[[-2.  -3.5]
 [ 2.   3.5]]
Infinitesimale Jackknife-Varianzschätzung: [ 8.  24.5]


In [62]:
T_N_star_mean = np.mean(T_N_b, axis=0).astype(np.float32)
print((N_bi[:, 0] - 1) )
(T_N_b[:, 0] - T_N_star_mean[0])

[ 1 -1  0]


array([-1.,  3., -2.])

In [108]:
print(N_bi[:, 0] - 165) 
print(T_N_b[:, 0] - T_N_star_mean[0])

print(np.dot(
    (N_bi[:, 0] - 6) , (T_N_b[:, 0] - T_N_star_mean[0])
).astype(np.float32) )

[-163 -165 -164]
[-1.  3. -2.]
-4.0


In [103]:
print(N_bi[:, 0] - 1) 
print(T_N_b[:, 0] - T_N_star_mean[0])

print(np.dot(
    (N_bi[:, 0] - 165) , (T_N_b[:, 0] - T_N_star_mean[0])
).astype(np.float32) )

[ 1 -1  0]
[-1.  3. -2.]
-4.0
